# QUESTION 5 (Heston Model, ρ = –0.30)

## sing the Heston Model and Monte-Carlo simulation, price an ATM European call and an ATM European put, using a correlation value of -0.30.

In [1]:
import numpy as np

# Base parameters
S0 = 80
K = 80
r = 0.055
T = 0.25
kappa = 1.85
theta = 0.045
v0 = 0.032
sigma = 0.35
rho = -0.30

N = 200      # time steps
M = 200000  # simulations


def simulate_heston(rho):
    np.random.seed(42)
    dt = T / N
    S = np.full(M, S0)
    v = np.full(M, v0)

    for _ in range(N):
        z1 = np.random.normal(size=M)
        z2 = rho * z1 + np.sqrt(1 - rho**2)*np.random.normal(size=M)

        # variance process
        v = np.abs(v + kappa*(theta - v)*dt + sigma*np.sqrt(np.maximum(v,0)) * np.sqrt(dt)*z1)

        # stock process
        S = S * np.exp((r - 0.5*v)*dt + np.sqrt(v*dt)*z2)

    return S


def price_heston(rho):
    ST = simulate_heston(rho)
    call = np.mean(np.maximum(ST - K, 0)) * np.exp(-r*T)
    put = np.mean(np.maximum(K - ST, 0)) * np.exp(-r*T)
    return call, put


call_5, put_5 = price_heston(rho=-0.30)
print(f"Call Price (rho = {rho}): {call_5:.2f}")
print(f"Put Price (rho = {rho}): {put_5:.2f}")

Call Price (rho = -0.3): 2.86
Put Price (rho = -0.3): 2.84


# QUESTION 6 (Heston Model, ρ = –0.70)

In [2]:
call_6, put_6 = price_heston(rho=-0.70)


print(f"Call Price (rho = -0.70): {call_6:.2f}")
print(f"Put Price (rho = -0.70): {put_6:.2f}")

Call Price (rho = -0.70): 2.12
Put Price (rho = -0.70): 3.45


# QUESTION 7 (Greeks for both Q5 and Q6)

## Calculate delta and gamma for each of the options in Questions 5 and 6. (Hint:You can numerically approximate this by forcing a change in the variable of interest –i.e., underlying stock price and delta change—and recalculating the option price).

In [3]:
def price_heston_with_S(S_new, rho):
    dt = T / N
    S = np.full(M, S_new)
    v = np.full(M, v0)

    for _ in range(N):
        z1 = np.random.normal(size=M)
        z2 = rho * z1 + np.sqrt(1 - rho**2)*np.random.normal(size=M)
        v = np.abs(v + kappa*(theta - v)*dt + sigma*np.sqrt(np.maximum(v,0))*np.sqrt(dt)*z1)
        S = S * np.exp((r - 0.5*v)*dt + np.sqrt(v*dt)*z2)

    call = np.mean(np.maximum(S - K, 0)) * np.exp(-r*T)
    put  = np.mean(np.maximum(K - S, 0)) * np.exp(-r*T)
    return call, put


def greeks_heston(rho, h=0.5):
    c_plus, p_plus = price_heston_with_S(S0 + h, rho)
    c_0, p_0 = price_heston_with_S(S0, rho)
    c_minus, p_minus = price_heston_with_S(S0 - h, rho)

    delta_c = (c_plus - c_minus) / (2*h)
    gamma_c = (c_plus - 2*c_0 + c_minus) / (h**2)

    delta_p = (p_plus - p_minus) / (2*h)
    gamma_p = (p_plus - 2*p_0 + p_minus) / (h**2)

    return delta_c, gamma_c, delta_p, gamma_p


# Greeks for Q5 and Q6
greeks_Q5 = greeks_heston(rho=-0.30)
greeks_Q6 = greeks_heston(rho=-0.70)

print("\n--- Greeks for Question 5 (rho = -0.30) ---")
print(f"Call Delta: {greeks_Q5[0]:.2f}")
print(f"Call Gamma: {greeks_Q5[1]:.2f}")
print(f"Put Delta: {greeks_Q5[2]:.2f}")
print(f"Put Gamma: {greeks_Q5[3]:.2f}")

print("\n--- Greeks for Question 6 (rho = -0.70) ---")
print(f"Call Delta: {greeks_Q6[0]:.2f}")
print(f"Call Gamma: {greeks_Q6[1]:.2f}")
print(f"Put Delta: {greeks_Q6[2]:.2f}")
print(f"Put Gamma: {greeks_Q6[3]:.2f}")


--- Greeks for Question 5 (rho = -0.30) ---
Call Delta: 0.52
Call Gamma: 0.11
Put Delta: -0.45
Put Gamma: 0.07

--- Greeks for Question 6 (rho = -0.70) ---
Call Delta: 0.48
Call Gamma: 0.03
Put Delta: -0.50
Put Gamma: 0.46


# Jump Modeler (Q8–Q10)

Here we implement the Merton jump–diffusion model to price ATM European call and put options and compute their Greeks (Delta and Gamma), as required in Questions 8–10.

We use:

- Spot price: \( $S_0$ = 80 \)
- Strike: \( $K$ = 80 \)
- Risk-free rate: \( $r$ = 5.5\% \)
- Volatility: \( $\sigma$ = 35\% \)
- Maturity: \( $T$ = 0.25 \)
- Jump mean: \( $\mu_J$ = -0.50 \)
- Jump volatility: \( $\delta_J$ = 0.22 \)

Jump intensities:
- Q8: \( $\lambda$ = 0.75 \)
- Q9: \( $\lambda$ = 0.25 \)

Q10 computes Delta and Gamma via centred finite differences.



In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

# Parameters
S0_base = 80.0
K_ATM = 80.0
r = 0.055
sigma = 0.35
T = 0.25
mu_J = -0.50
delta_J = 0.22

N_PATHS = 100_000
seed_base = 12345


def simulate_merton_terminal(S0, r, sigma, T, lam, mu_J, delta_J, n_paths, rng):
    """
    Simulate terminal stock prices under Merton's jump–diffusion.
    """
    kappa = np.exp(mu_J + 0.5 * delta_J**2) - 1
    drift = (r - 0.5 * sigma**2 - lam * kappa) * T

    Z = rng.normal(size=n_paths)
    N = rng.poisson(lam * T, size=n_paths)

    jump_mean = N * mu_J
    jump_std = np.sqrt(N) * delta_J
    jump_std[N == 0] = 0.0

    Y_sum = jump_mean + jump_std * rng.normal(size=n_paths)

    log_ST = np.log(S0) + drift + sigma * np.sqrt(T) * Z + Y_sum
    return np.exp(log_ST)


def price_european_merton(S0, K, r, sigma, T, lam, mu_J, delta_J,
                          n_paths, option_type="call", seed=None):
    """
    Monte Carlo pricing for European call/put under Merton model.
    """
    rng = np.random.default_rng(seed)
    ST = simulate_merton_terminal(S0, r, sigma, T, lam, mu_J, delta_J,
                                  n_paths, rng)
    discount = np.exp(-r*T)

    if option_type == "call":
        payoff = np.maximum(ST - K, 0)
    else:
        payoff = np.maximum(K - ST, 0)

    pv = discount * payoff
    return pv.mean(), pv.std(ddof=1)/np.sqrt(n_paths)


## Q8 — Pricing ATM European Call & Put for λ = 0.75


In [5]:
lam_8 = 0.75

call_8_price, call_8_se = price_european_merton(
    S0_base, K_ATM, r, sigma, T, lam_8,
    mu_J, delta_J, N_PATHS, "call", seed_base
)

put_8_price, put_8_se = price_european_merton(
    S0_base, K_ATM, r, sigma, T, lam_8,
    mu_J, delta_J, N_PATHS, "put", seed_base+10
)

print("Q8 Results (λ = 0.75):")
print(f"Call Price ≈ {call_8_price:.4f}   SE = {call_8_se:.4f}")
print(f"Put Price  ≈ {put_8_price:.4f}   SE = {put_8_se:.4f}")


Q8 Results (λ = 0.75):
Call Price ≈ 8.3874   SE = 0.0364
Put Price  ≈ 7.2393   SE = 0.0389


## Q9 — Pricing ATM European Call & Put for λ = 0.25


In [6]:
lam_9 = 0.25

call_9_price, call_9_se = price_european_merton(
    S0_base, K_ATM, r, sigma, T, lam_9,
    mu_J, delta_J, N_PATHS, "call", seed_base+100
)

put_9_price, put_9_se = price_european_merton(
    S0_base, K_ATM, r, sigma, T, lam_9,
    mu_J, delta_J, N_PATHS, "put", seed_base+110
)

print("Q9 Results (λ = 0.25):")
print(f"Call Price ≈ {call_9_price:.4f}   SE = {call_9_se:.4f}")
print(f"Put Price  ≈ {put_9_price:.4f}   SE = {put_9_se:.4f}")

pd.DataFrame({
    "λ": [0.75, 0.25],
    "Call Price": [call_8_price, call_9_price],
    "Put Price": [put_8_price, put_9_price]
})


Q9 Results (λ = 0.25):
Call Price ≈ 6.8171   SE = 0.0319
Put Price  ≈ 5.7499   SE = 0.0293


,λ,Call Price,Put Price
0,0.75,8.387376,7.239328
1,0.25,6.817052,5.749921


## Q10 — Delta and Gamma via Finite Differences
We compute:
$\Delta \approx \dfrac{V(S_0+0.5)-V(S_0-0.5)}{2 \cdot 0.5}, \qquad
\Gamma \approx \dfrac{V(S_0+0.5)-2V(S_0)+V(S_0-0.5)}{(0.5)^2}$



In [7]:
def greek_estimates_merton(S0, K, lam, option_type, h=0.5, seed_base=900):
    V0, _ = price_european_merton(S0, K, r, sigma, T, lam,
                                  mu_J, delta_J, N_PATHS, option_type, seed_base)
    V_up, _ = price_european_merton(S0+h, K, r, sigma, T, lam,
                                    mu_J, delta_J, N_PATHS, option_type, seed_base+1)
    V_down, _ = price_european_merton(S0-h, K, r, sigma, T, lam,
                                      mu_J, delta_J, N_PATHS, option_type, seed_base+2)

    delta = (V_up - V_down)/(2*h)
    gamma = (V_up - 2*V0 + V_down)/(h*h)
    return V0, delta, gamma

rows = []

for lam_val, label in [(lam_8, "λ = 0.75"), (lam_9, "λ = 0.25")]:
    for option_type in ["call", "put"]:
        V0, d, g = greek_estimates_merton(S0_base, K_ATM, lam_val,
                                         option_type, h=0.5)
        rows.append([label, option_type.capitalize(), V0, d, g])

greeks_df = pd.DataFrame(rows, columns=["Jump Intensity", "Option Type",
                                        "Price", "Delta", "Gamma"])
greeks_df


,Jump Intensity,Option Type,Price,Delta,Gamma
0,λ = 0.75,Call,8.348575,0.644433,-0.690798
1,λ = 0.75,Put,7.229880,-0.391607,0.042684
2,λ = 0.25,Call,6.853056,0.606781,-0.513106
3,λ = 0.25,Put,5.711196,-0.408813,0.127574


# QUESTION 11 (Put-Call Parity Check)

In [8]:
import numpy as np
import pandas as pd

# ==========================================
# 1. Parameter Setup
# ==========================================
# General Parameters
S0 = 80.0          # Initial Stock Price
r = 0.055          # Risk-free rate (5.5%)
T = 0.25           # Time to maturity (3 months)
sigma_gen = 0.35   # General sigma

# Heston Model Parameters
v0 = 0.032         # Initial variance
kappa = 1.85       # Mean reversion speed
theta = 0.045      # Long-run variance

# Merton Model Parameters
mu_j = -0.5        # Mean jump size
delta_j = 0.22     # Jump standard deviation

# Simulation Settings
N_SIMS = 100000    # Number of simulations
N_STEPS = 90       # Daily steps
dt = T / N_STEPS

np.random.seed(42) # Set seed for reproducibility

# ==========================================
# 2. Model Functions
# ==========================================

def heston_mc_price(S0, K, r, T, v0, kappa, theta, sigma_v, rho, option_type='call'):
    # Generate correlated Brownian motions
    Z1 = np.random.normal(size=(N_SIMS, N_STEPS))
    Z2 = rho * Z1 + np.sqrt(1 - rho**2) * np.random.normal(size=(N_SIMS, N_STEPS))

    S = np.zeros((N_SIMS, N_STEPS + 1))
    v = np.zeros((N_SIMS, N_STEPS + 1))
    S[:, 0] = S0
    v[:, 0] = v0

    for t in range(N_STEPS):
        # Full truncation for variance
        v_curr = np.maximum(v[:, t], 0)

        # Variance Process
        dv = kappa * (theta - v_curr) * dt + sigma_v * np.sqrt(v_curr * dt) * Z2[:, t]
        v[:, t+1] = v[:, t] + dv

        # Price Process
        dS = r * S[:, t] * dt + np.sqrt(v_curr * dt) * S[:, t] * Z1[:, t]
        S[:, t+1] = S[:, t] + dS

    # Calculate Payoff
    ST = S[:, -1]
    if option_type == 'call':
        payoff = np.maximum(ST - K, 0)
    elif option_type == 'put':
        payoff = np.maximum(K - ST, 0)

    return np.exp(-r * T) * np.mean(payoff)

def merton_mc_price(S0, K, r, T, sigma, lam, mu_j, delta_j, option_type='call'):
    # Expected jump size (k)
    k = np.exp(mu_j + 0.5 * delta_j**2) - 1

    # Simulate number of jumps
    n_jumps = np.random.poisson(lam * T, N_SIMS)

    # Simulate standard Brownian motion component
    Z = np.random.normal(0, 1, N_SIMS)

    # Simulate Jump sizes
    jump_component = np.random.normal(n_jumps * mu_j, np.sqrt(n_jumps) * delta_j, N_SIMS)

    # Drift and Diffusion components
    drift = (r - lam * k - 0.5 * sigma**2) * T
    diffusion = sigma * np.sqrt(T) * Z

    # Stock price at T
    ST = S0 * np.exp(drift + diffusion + jump_component)

    # Calculate Payoff
    if option_type == 'call':
        payoff = np.maximum(ST - K, 0)
    elif option_type == 'put':
        payoff = np.maximum(K - ST, 0)

    return np.exp(-r * T) * np.mean(payoff)

# ==========================================
# 3. Execution for Question 11 (Parity Check)
# ==========================================
print("--- Question 11: Put-Call Parity Check ---")

scenarios = [
    {"Model": "Heston (Q5)", "rho": -0.30, "lam": None, "func": heston_mc_price},
    {"Model": "Heston (Q6)", "rho": -0.70, "lam": None, "func": heston_mc_price},
    {"Model": "Merton (Q8)", "rho": None, "lam": 0.75, "func": merton_mc_price},
    {"Model": "Merton (Q9)", "rho": None, "lam": 0.25, "func": merton_mc_price},
]

parity_results = []

for sc in scenarios:
    if sc["Model"].startswith("Heston"):
        c_price = heston_mc_price(S0, S0, r, T, v0, kappa, theta, sigma_gen, sc["rho"], 'call')
        p_price = heston_mc_price(S0, S0, r, T, v0, kappa, theta, sigma_gen, sc["rho"], 'put')
    else:
        c_price = merton_mc_price(S0, S0, r, T, sigma_gen, sc["lam"], mu_j, delta_j, 'call')
        p_price = merton_mc_price(S0, S0, r, T, sigma_gen, sc["lam"], mu_j, delta_j, 'put')

    lhs = c_price - p_price
    rhs = S0 - S0 * np.exp(-r * T)
    diff = lhs - rhs

    parity_results.append({
        "Model Case": sc["Model"],
        "Call Price": round(c_price, 2),
        "Put Price": round(p_price, 2),
        "LHS (C-P)": round(lhs, 4),
        "RHS (S-Ke^-rT)": round(rhs, 4),
        "Difference": round(diff, 4),
        "Parity Holds?": "Yes" if abs(diff) < 0.1 else "Check"
    })

df_parity = pd.DataFrame(parity_results)
print(df_parity)
print("\n")

--- Question 11: Put-Call Parity Check ---
    Model Case  Call Price  Put Price  LHS (C-P)  RHS (S-Ke^-rT)  Difference  \
0  Heston (Q5)        3.46       2.38     1.0802          1.0925     -0.0122   
1  Heston (Q6)        3.49       2.42     1.0763          1.0925     -0.0162   
2  Merton (Q8)        8.28       7.17     1.1118          1.0925      0.0193   
3  Merton (Q9)        6.85       5.74     1.1019          1.0925      0.0095   

  Parity Holds?  
0           Yes  
1           Yes  
2           Yes  
3           Yes  




# QUESTION 12 (7 Different Strikes)

In [9]:
print("--- Question 12: 7 Different Strikes ---")

# Moneyness = Stock / Strike
moneyness_levels = [0.85, 0.90, 0.95, 1.00, 1.05, 1.10, 1.15]
strikes = [S0 / m for m in moneyness_levels]

# Representative parameters for Q12
rho_q12 = -0.70
lam_q12 = 0.75

strike_results = []

for m, K in zip(moneyness_levels, strikes):
    # Heston Call
    h_call = heston_mc_price(S0, K, r, T, v0, kappa, theta, sigma_gen, rho_q12, 'call')
    # Merton Call
    m_call = merton_mc_price(S0, K, r, T, sigma_gen, lam_q12, mu_j, delta_j, 'call')

    strike_results.append({
        "Moneyness (S/K)": m,
        "Strike (K)": round(K, 2),
        "Heston Call Price": round(h_call, 2),
        "Merton Call Price": round(m_call, 2)
    })

df_strikes = pd.DataFrame(strike_results)
print(df_strikes)

--- Question 12: 7 Different Strikes ---
   Moneyness (S/K)  Strike (K)  Heston Call Price  Merton Call Price
0             0.85       94.12               0.04               2.83
1             0.90       88.89               0.38               4.38
2             0.95       84.21               1.49               6.17
3             1.00       80.00               3.51               8.32
4             1.05       76.19               6.05              10.61
5             1.10       72.73               8.87              12.85
6             1.15       69.57              11.66              15.09


# QUESTION 13 (American call - reuse Q5 & Q7 code)

In [10]:

# - price_heston(rho) -> returns (call_price, put_price)
# - greeks_heston(rho, h=0.5) -> returns (delta_call, gamma_call, delta_put, gamma_put)


# 1) Price (re-uses European price computed by the  price_heston(rho) function)
call_price, _ = price_heston(rho=-0.30)

# 2) Greeks (greeks_heston returns four values )
delta_call, gamma_call, delta_put, gamma_put = greeks_heston(rho=-0.30, h=0.5)




print("Question 13 — American ATM call (Heston, rho = -0.30)")
print("---------------------------------------------------")
print(f"Model note: Non-dividend stock => American call = European call (no early-exercise premium).")
print(f"Call price (rounded to cents): ${call_price:.2f}")
print(f"Call Delta (approx): {delta_call:.2f}")
print(f"Call Gamma (approx): {gamma_call:.2f}")




Question 13 — American ATM call (Heston, rho = -0.30)
---------------------------------------------------
Model note: Non-dividend stock => American call = European call (no early-exercise premium).
Call price (rounded to cents): $10.61
Call Delta (approx): 0.91
Call Gamma (approx): 0.01


# QUESTION 14 (Heston Up-and In Call)

In [11]:
# ---------------------------------------------------------
# Q14. Heston Up-and-In Call (CUI)
# Parameters: S0=80, K=95, Barrier(H)=95, rho=-0.7 (from Q6)
# ---------------------------------------------------------
print("Running Q14: Heston Up-and-In Call...")

# Q6 Parameters
rho_q14 = -0.70
K_q14 = 95.0
H_q14 = 95.0

# Heston Simulation with Path Monitoring
# Re-using logic but calculating Max price along path
Z1 = np.random.normal(size=(N_SIMS, N_STEPS))
Z2 = rho_q14 * Z1 + np.sqrt(1 - rho_q14**2) * np.random.normal(size=(N_SIMS, N_STEPS))

S = np.zeros((N_SIMS, N_STEPS + 1))
v = np.zeros((N_SIMS, N_STEPS + 1))
S[:, 0] = S0
v[:, 0] = v0

for t in range(N_STEPS):
    v_curr = np.maximum(v[:, t], 0)
    dv = kappa * (theta - v_curr) * dt + sigma_gen * np.sqrt(v_curr * dt) * Z2[:, t]
    v[:, t+1] = v[:, t] + dv

    dS = r * S[:, t] * dt + np.sqrt(v_curr * dt) * S[:, t] * Z1[:, t]
    S[:, t+1] = S[:, t] + dS

# 1. Vanilla European Call Price (at K=95)
ST = S[:, -1]
payoff_vanilla = np.maximum(ST - K_q14, 0)
price_vanilla_q14 = np.exp(-r * T) * np.mean(payoff_vanilla)

# 2. Up-and-In Call Price
# Check if Barrier H=95 was hit at any time step
# max over axis 1 (time steps)
path_max = np.max(S, axis=1)
barrier_hit = path_max >= H_q14

# Payoff is vanilla payoff ONLY IF barrier was hit
payoff_ui = np.where(barrier_hit, payoff_vanilla, 0)
price_ui_q14 = np.exp(-r * T) * np.mean(payoff_ui)

print(f"Q14 Heston Vanilla Call (K={K_q14}): {price_vanilla_q14:.2f}")
print(f"Q14 Heston Up-and-In Call (H={H_q14}, K={K_q14}): {price_ui_q14:.2f}")
print("-" * 30)

Running Q14: Heston Up-and-In Call...
Q14 Heston Vanilla Call (K=95.0): 0.03
Q14 Heston Up-and-In Call (H=95.0, K=95.0): 0.03
------------------------------


# QUESTION 15 (Merton Down-and-In Put)

In [12]:
# ---------------------------------------------------------
# Q15. Merton Down-and-In Put (PDI)
# Parameters: S0=80, K=65, Barrier(H)=65, lambda=0.75 (from Q8)
# ---------------------------------------------------------
print("Running Q15: Merton Down-and-In Put...")

# Q8 Parameters
lam_q15 = 0.75
K_q15 = 65.0
H_q15 = 65.0

# Merton Simulation with Path Monitoring
# Needs full path simulation unlike the optimized endpoint-only version
k_j = np.exp(mu_j + 0.5 * delta_j**2) - 1
drift_m = (r - lam_q15 * k_j - 0.5 * sigma_gen**2) * dt

S_m = np.zeros((N_SIMS, N_STEPS + 1))
S_m[:, 0] = S0

for t in range(N_STEPS):
    Z = np.random.normal(0, 1, N_SIMS)

    # Poisson Jumps (Bernoulli approx for small dt is simpler,
    # but using Poisson random number is more accurate)
    # Number of jumps in this step
    N_jumps = np.random.poisson(lam_q15 * dt, N_SIMS)

    # Sum of jumps sizes
    # If N=0, sum is 0. If N>0, sum N normal variables.
    # Vectorized approach: approx standard deviation based on N
    # J_sum ~ N(N*mu, N*delta^2)

    jump_mean = N_jumps * mu_j
    jump_std = np.sqrt(N_jumps) * delta_j
    # Handle case where N=0 -> std=0
    jump_val = np.random.normal(jump_mean, np.where(N_jumps > 0, jump_std, 0))

    diffusion = sigma_gen * np.sqrt(dt) * Z

    # dlnS = (r - lam*k - 0.5*sigma^2)dt + sigma*dW + dJ
    # S_t+1 = S_t * exp(...)
    S_m[:, t+1] = S_m[:, t] * np.exp(drift_m + diffusion + jump_val)

# 1. Vanilla European Put Price (at K=65)
ST_m = S_m[:, -1]
payoff_vanilla_m = np.maximum(K_q15 - ST_m, 0)
price_vanilla_q15 = np.exp(-r * T) * np.mean(payoff_vanilla_m)

# 2. Down-and-In Put Price
# Check if Barrier H=65 was hit (crossed below)
path_min = np.min(S_m, axis=1)
barrier_hit_m = path_min <= H_q15

payoff_di = np.where(barrier_hit_m, payoff_vanilla_m, 0)
price_di_q15 = np.exp(-r * T) * np.mean(payoff_di)

print(f"Q15 Merton Vanilla Put (K={K_q15}): {price_vanilla_q15:.2f}")
print(f"Q15 Merton Down-and-In Put (H={H_q15}, K={K_q15}): {price_di_q15:.2f}")

Running Q15: Merton Down-and-In Put...
Q15 Merton Vanilla Put (K=65.0): 2.73
Q15 Merton Down-and-In Put (H=65.0, K=65.0): 2.73
